In [2]:
import os

print(" Downloaden van de Microsoft Repository...")
!git clone https://github.com/microsoft/MedImaging-ModelDriftMonitoring.git

if os.path.exists("MedImaging-ModelDriftMonitoring"):
    print("Repository is succesvol binnengehaald!")
else:
    print(" Er is iets misgegaan met downloaden.")

 Downloaden van de Microsoft Repository...
fatal: destination path 'MedImaging-ModelDriftMonitoring' already exists and is not an empty directory.
Repository is succesvol binnengehaald!


In [3]:
# Installeer de benodigde pakketten met de juiste versies (geen Autocast errors meer!)
%pip install torch torchvision pandas scikit-image torchxrayvision plotly scipy tqdm torchprof environs param jupyter opencv-python-headless torchmetrics mlflow tensorboard pytorch-lightning==1.9.5 lightning-bolts==0.7.0 sklearn-pandas
%pip install azureml-core

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
%%writefile MedImaging-ModelDriftMonitoring/src/scripts/vae/train.py
import os, argparse, torch, sys
import pandas as pd, numpy as np, skimage.io
import torchxrayvision as xrv
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '../../..')))
from src.model_drift.models.vae import VAE

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--images-dir', type=str)
    parser.add_argument('--tsv-file', type=str)
    parser.add_argument('--output-dir', type=str)
    parser.add_argument('--epochs', type=int, default=50)
    parser.add_argument('--batch-size', type=int, default=4)
    args, _ = parser.parse_known_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    df_orig = pd.read_csv(args.tsv_file, sep='\t', low_memory=False, dtype={'ImageID': str})
    df_orig['StudyDate_DICOM'] = pd.to_datetime(df_orig['StudyDate_DICOM'], format='%Y%m%d', errors='coerce')

    try: bestanden = set(os.listdir(args.images_dir))
    except: bestanden = set()

    def get_pad(img_id):
        if pd.isna(img_id): return None
        base_id = str(img_id).replace('.png', '').replace('.jpg', '').strip()
        for optie in [f"{base_id}.png", f"{base_id}.jpg", str(img_id).strip()]:
            if optie in bestanden: return os.path.join(args.images_dir, optie)
        return None

    df_orig['Echt_Pad'] = df_orig['ImageID'].apply(get_pad)
    df = df_orig.dropna(subset=['StudyDate_DICOM', 'ImageID', 'Echt_Pad']).reset_index(drop=True)
    df_ref = df[df['StudyDate_DICOM'].dt.year <= 2015].copy().reset_index(drop=True)

    class VAEDataset(Dataset):
        def __init__(self, df):
            self.df = df
            self.resizer = xrv.datasets.XRayResizer(128)
        def __len__(self): return len(self.df)
        def __getitem__(self, idx):
            try:
                img = skimage.io.imread(self.df.iloc[idx]['Echt_Pad'], as_gray=True).astype(np.float32)
                if img.ndim != 2: img = img[:, :, 0]
            except: img = np.zeros((128, 128), dtype=np.float32)

            if img.max() <= 1.0: img = xrv.datasets.normalize(img, 1.0)
            elif img.max() <= 255.0: img = xrv.datasets.normalize(img, 255.0)
            else: img = xrv.datasets.normalize(img, 65535.0)
            return torch.from_numpy(self.resizer(img[np.newaxis, ...])).float()

    loader = DataLoader(VAEDataset(df_ref), batch_size=min(args.batch_size, max(1, len(df_ref))), shuffle=True)
    vae = VAE(image_dims=(1, 128, 128), zsize=128, kl_coeff=0.1, log_recon_images=0).to(device)
    optimizer = torch.optim.Adam(vae.parameters(), lr=1e-4)

    vae.train()
    for epoch in range(args.epochs):
        loss_sum = 0
        for batch in tqdm(loader, desc=f"Epoch {epoch+1}/{args.epochs}"):
            optimizer.zero_grad()
            loss, _, _ = vae.step(batch.to(device))
            loss.backward()
            optimizer.step()
            loss_sum += loss.item()
        print(f"Epoch {epoch+1} Loss: {loss_sum/max(1, len(loader)):.4f}")

    os.makedirs(args.output_dir, exist_ok=True)
    torch.save({'state_dict': vae.state_dict()}, os.path.join(args.output_dir, 'vae_weights.ckpt'))

if __name__ == '__main__': main()

Overwriting MedImaging-ModelDriftMonitoring/src/scripts/vae/train.py


In [5]:
%%writefile MedImaging-ModelDriftMonitoring/src/scripts/vae/score.py
import os, argparse, torch, sys
import pandas as pd, numpy as np, skimage.io
import torchxrayvision as xrv
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '../../..')))
from src.model_drift.models.vae import VAE

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--images-dir', type=str)
    parser.add_argument('--tsv-file', type=str)
    parser.add_argument('--model-path', type=str)
    parser.add_argument('--output-dir', type=str)
    parser.add_argument('--batch-size', type=int, default=4)
    args, _ = parser.parse_known_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    df = pd.read_csv(args.tsv_file, sep='\t', low_memory=False, dtype={'ImageID': str})
    
    try: bestanden = set(os.listdir(args.images_dir))
    except: bestanden = set()

    def get_pad(img_id):
        if pd.isna(img_id): return None
        base_id = str(img_id).replace('.png', '').replace('.jpg', '').strip()
        for optie in [f"{base_id}.png", f"{base_id}.jpg", str(img_id).strip()]:
            if optie in bestanden: return os.path.join(args.images_dir, optie)
        return None
        
    df['Echt_Pad'] = df['ImageID'].apply(get_pad)
    df = df.dropna(subset=['ImageID', 'Echt_Pad']).reset_index(drop=True)

    class VAEScoreData(Dataset):
        def __init__(self, df):
            self.df = df
            self.resizer = xrv.datasets.XRayResizer(128)
        def __len__(self): return len(self.df)
        def __getitem__(self, idx):
            try:
                img = skimage.io.imread(self.df.iloc[idx]['Echt_Pad'], as_gray=True).astype(np.float32)
                if img.ndim != 2: img = img[:, :, 0]
            except: img = np.zeros((128, 128), dtype=np.float32)

            if img.max() <= 1.0: img = xrv.datasets.normalize(img, 1.0)
            elif img.max() <= 255.0: img = xrv.datasets.normalize(img, 255.0)
            else: img = xrv.datasets.normalize(img, 65535.0)
            return torch.from_numpy(self.resizer(img[np.newaxis, ...])).float(), self.df.iloc[idx]['ImageID']

    loader = DataLoader(VAEScoreData(df), batch_size=args.batch_size, shuffle=False, drop_last=True)
    
    vae = VAE(image_dims=(1, 128, 128), zsize=128, kl_coeff=0.1, log_recon_images=0).to(device)
    vae.load_state_dict(torch.load(args.model_path, map_location=device)['state_dict'])
    vae.eval()

    results = []
    with torch.no_grad():
        for batch_img, batch_ids in tqdm(loader, desc="VAE Extractie"):
            out = vae(batch_img.to(device))
            mu = (out[2] if len(out)==4 else out[1]).cpu().numpy()
            for img_id, latents in zip(batch_ids, mu):
                row = {'ImageID': img_id.item() if hasattr(img_id, 'item') else img_id}
                for i in range(128): row[f'mu.{i:03d}'] = latents[i]
                results.append(row)
                
    os.makedirs(args.output_dir, exist_ok=True)
    pd.DataFrame(results).to_csv(os.path.join(args.output_dir, 'vae_scores.csv'), index=False)

if __name__ == '__main__': main()

Overwriting MedImaging-ModelDriftMonitoring/src/scripts/vae/score.py


In [6]:
%%writefile MedImaging-ModelDriftMonitoring/src/scripts/finetune/score.py
import os, argparse, torch, sys
import pandas as pd, numpy as np, skimage.io
import torchxrayvision as xrv
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--images-dir', type=str)
    parser.add_argument('--tsv-file', type=str)
    parser.add_argument('--output-dir', type=str)
    parser.add_argument('--batch-size', type=int, default=4)
    args, _ = parser.parse_known_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    df = pd.read_csv(args.tsv_file, sep='\t', low_memory=False, dtype={'ImageID': str})
    
    try: bestanden = set(os.listdir(args.images_dir))
    except: bestanden = set()

    def get_pad(img_id):
        if pd.isna(img_id): return None
        base_id = str(img_id).replace('.png', '').replace('.jpg', '').strip()
        for optie in [f"{base_id}.png", f"{base_id}.jpg", str(img_id).strip()]:
            if optie in bestanden: return os.path.join(args.images_dir, optie)
        return None
        
    df['Echt_Pad'] = df['ImageID'].apply(get_pad)
    df = df.dropna(subset=['ImageID', 'Echt_Pad']).reset_index(drop=True)

    class ClfScoreData(Dataset):
        def __init__(self, df):
            self.df = df
            self.resizer = xrv.datasets.XRayResizer(224)
        def __len__(self): return len(self.df)
        def __getitem__(self, idx):
            try:
                img = skimage.io.imread(self.df.iloc[idx]['Echt_Pad'], as_gray=True).astype(np.float32)
                if img.ndim != 2: img = img[:, :, 0]
            except: img = np.zeros((224, 224), dtype=np.float32)

            if img.max() <= 1.0: img = xrv.datasets.normalize(img, 1.0)
            elif img.max() <= 255.0: img = xrv.datasets.normalize(img, 255.0)
            else: img = xrv.datasets.normalize(img, 65535.0)
            return torch.from_numpy(self.resizer(img[np.newaxis, ...])).float(), self.df.iloc[idx]['ImageID']


    loader = DataLoader(ClfScoreData(df), batch_size=args.batch_size, shuffle=False, drop_last=True)
    
    classifier = xrv.models.DenseNet(weights="densenet121-res224-all").to(device)
    classifier.eval()

    results = []
    with torch.no_grad():
        for batch_img, batch_ids in tqdm(loader, desc="DenseNet Extractie"):
            pred = classifier(batch_img.to(device))
            pneum_idx = classifier.pathologies.index("Pneumonia")
            preds = pred[:, pneum_idx].cpu().numpy()
            for img_id, p in zip(batch_ids, preds):
                results.append({'ImageID': img_id.item() if hasattr(img_id, 'item') else img_id, 'prob_Pneumonia': p})
                
    os.makedirs(args.output_dir, exist_ok=True)
    pd.DataFrame(results).to_csv(os.path.join(args.output_dir, 'clf_scores.csv'), index=False)

if __name__ == '__main__': main()

Overwriting MedImaging-ModelDriftMonitoring/src/scripts/finetune/score.py


In [7]:
%%writefile MedImaging-ModelDriftMonitoring/src/scripts/drift/generate_drift_csv.py
import os, argparse, sys
import pandas as pd, numpy as np
from scipy.stats import ks_2samp, chi2_contingency
from sklearn.metrics import roc_auc_score

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '../../..')))
from src.model_drift.figure_helper import FigureHelper

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--tsv-file', type=str)
    parser.add_argument('--vae-scores', type=str)
    parser.add_argument('--clf-scores', type=str)
    parser.add_argument('--weights-file', type=str)
    parser.add_argument('--output-dir', type=str)
    args, _ = parser.parse_known_args()

    # --- 1. DATA INLADEN ---
    df_meta = pd.read_csv(args.tsv_file, sep='\t', low_memory=False)
    df_vae = pd.read_csv(args.vae_scores, dtype={'ImageID': str})
    df_clf = pd.read_csv(args.clf_scores, dtype={'ImageID': str})

    for d in [df_meta, df_vae, df_clf]:
        d['ImageID'] = d['ImageID'].astype(str).str.replace('.png', '', regex=False).str.replace('.jpg', '', regex=False)

    df = pd.merge(df_meta, df_vae, on='ImageID', how='inner')
    df = pd.merge(df, df_clf, on='ImageID', how='inner')
    df['StudyDate_DICOM'] = pd.to_datetime(df['StudyDate_DICOM'], format='%Y%m%d', errors='coerce')
    df = df.dropna(subset=['StudyDate_DICOM']).sort_values('StudyDate_DICOM').reset_index(drop=True)

    # Definieer Metadata Kolommen
    meta_cont_kandidaat = ['age', 'Columns_DICOM', 'Rows_DICOM']
    meta_cat_kandidaat = ['PatientSex', 'ViewPosition', 'Manufacturer', 'Modality']
    
    meta_cont = [c for c in meta_cont_kandidaat if c in df.columns]
    meta_cat = [c for c in meta_cat_kandidaat if c in df.columns]

    if 'Labels' not in df.columns:
        print(" WAARSCHUWING: Kolom 'Labels' niet gevonden! AUROC kan niet berekend worden.")
    
    # Chronologische split (30% baseline)
    split_idx = int(len(df) * 0.30)
    df_ref = df.iloc[:split_idx].copy()
    df_prod = df.iloc[split_idx:].copy()
    
    weights_dict = {}
    if os.path.exists(args.weights_file):
        df_weights = pd.read_csv(args.weights_file)
        if 'Unnamed: 2' in df_weights.columns:
            df_weights = df_weights[df_weights['Unnamed: 2'] == 'distance']
            weights_dict = dict(zip(df_weights['Unnamed: 0'], df_weights['abs(corr)']))

    # Kolommen voor de algemene MMC
    kolommen = ['prob_Pneumonia'] + meta_cont + [f'mu.{i:03d}' for i in range(128)]
    beschikbare_features = [c for c in kolommen if c in df.columns]

    print(f" Analyse opstarten... Baseline N={len(df_ref)}, Productie N={len(df_prod)}")
    
    # --- 2. BASELINE BEREKENEN VOOR MMC ---
    baseline = {}
    FIXED_SAMPLE_SIZE = min(50, len(df_ref) // 3) 
    bootstraps = {c: [] for c in beschikbare_features}
    
    for _ in range(50):
        samp = df_ref.sample(n=FIXED_SAMPLE_SIZE, replace=True)
        for c in beschikbare_features:
            d_ref, d_samp = df_ref[c].dropna(), samp[c].dropna()
            if len(d_ref) > 10 and len(d_samp) > 10:
                stat, _ = ks_2samp(d_ref, d_samp)
                bootstraps[c].append(stat)
                
    for c, dists in bootstraps.items():
        if dists: 
            baseline[c] = {'mean': np.mean(dists), 'std': max(np.std(dists), 0.02)}

    # --- 3. DRIFT & PRESTATIES BEREKENEN PER MAAND ---
    results_mmc, results_auroc = [], []
    tracking_ks = []
    tracking_chi2 = []
    
    # BASELINE ANCHORS (Teruggezet uit jouw werkende code!)
    anchor_date = df_ref['StudyDate_DICOM'].max()
    if not pd.isna(anchor_date):
        results_mmc.append({'Datum': anchor_date, 'MMC': 0.0}) 
        
        if 'Labels' in df_ref.columns and 'prob_Pneumonia' in df_ref.columns:
            y_true_ref = df_ref['Labels'].apply(lambda x: 1 if str(x).strip() == '1' or 'pneumonia' in str(x).lower() else 0)
            if len(y_true_ref.unique()) > 1:
                ref_auc = roc_auc_score(y_true_ref, df_ref['prob_Pneumonia'])
                results_auroc.append({'Datum': anchor_date, 'AUROC_Score': ref_auc})

    maanden = [g for n, g in df_prod.groupby(pd.Grouper(key='StudyDate_DICOM', freq='M'))]
    
    for i in range(len(maanden)):
        if i == 0: win_df = maanden[i]
        elif i == 1: win_df = pd.concat([maanden[i-1], maanden[i]])
        else: win_df = pd.concat([maanden[i-2], maanden[i-1], maanden[i]])
            
        datum = maanden[i]['StudyDate_DICOM'].max()
        if pd.isna(datum) or len(win_df) < 30:
            continue

        # A) MMC BEREKENEN 
        teller, noemer = 0.0, 0.0
        for c in baseline.keys():
            if c in win_df.columns:
                stats = []
                d_win = win_df[c].dropna()
                for _ in range(10):
                    if len(d_win) >= FIXED_SAMPLE_SIZE:
                        samp = d_win.sample(n=FIXED_SAMPLE_SIZE, replace=True)
                        s, _ = ks_2samp(df_ref[c].dropna(), samp)
                        stats.append(s)
                if stats:
                    gemiddelde_stat = np.mean(stats)
                    z_score = (gemiddelde_stat - baseline[c]['mean']) / baseline[c]['std']
                    z_score = np.clip(z_score, -3.0, 3.0)
                    gew = weights_dict.get(c, 0.1)
                    teller += gew * z_score
                    noemer += gew
                    
        mmc_score = -(teller/noemer) if noemer > 0 else 0
        results_mmc.append({'Datum': datum, 'MMC': mmc_score})

        # B) AUROC BEREKENEN
        if 'Labels' in win_df.columns and 'prob_Pneumonia' in win_df.columns:
            y_true = win_df['Labels'].apply(lambda x: 1 if str(x).strip() == '1' or 'pneumonia' in str(x).lower() else 0)
            y_pred = win_df['prob_Pneumonia'].astype(float)
            
            if len(y_true.unique()) > 1:
                auc = roc_auc_score(y_true, y_pred)
                results_auroc.append({'Datum': datum, 'AUROC_Score': auc})

        # C) TRANSPARANTE METADATA DRIFT (EXPORTEREN NAAR MAPJE)
        maand_ks = {'Datum': datum}
        for c in meta_cont:
            d_ref, d_win = df_ref[c].dropna(), win_df[c].dropna()
            if len(d_ref) > 10 and len(d_win) > 10:
                stat, pval = ks_2samp(d_ref, d_win)
                maand_ks[f'{c}_KS_Score'] = stat
                maand_ks[f'{c}_P_Value'] = pval
        tracking_ks.append(maand_ks)

        maand_chi2 = {'Datum': datum}
        for c in meta_cat:
            ref_counts = df_ref[c].value_counts()
            win_counts = win_df[c].value_counts()
            all_cats = list(set(ref_counts.index) | set(win_counts.index))
            
            ref_freq = [ref_counts.get(cat, 0) for cat in all_cats]
            win_freq = [win_counts.get(cat, 0) for cat in all_cats]
            
            # Alleen berekenen als er spreiding is
            if sum(ref_freq) > 0 and sum(win_freq) > 0:
                try:
                    chi2_stat, pval, _, _ = chi2_contingency([ref_freq, win_freq])
                    maand_chi2[f'{c}_Chi2_Score'] = chi2_stat
                    maand_chi2[f'{c}_P_Value'] = pval
                except ValueError:
                    pass
        tracking_chi2.append(maand_chi2)

    # --- 4. OPSLAAN & VISUALISEREN ---
    os.makedirs(args.output_dir, exist_ok=True)
    
    if len(results_mmc) > 1:
        df_mmc = pd.DataFrame(results_mmc).set_index('Datum')
        df_mmc['MMC_Smoothed'] = df_mmc['MMC'].ewm(span=4, adjust=False).mean()
        # CSV export is weer terug!
        df_mmc.to_csv(os.path.join(args.output_dir, 'mmc_results_smoothed.csv'))
        
        h_mmc = FigureHelper(x=df_mmc.index)
        h_mmc.add_trace(y=df_mmc['MMC_Smoothed'], name='Stabiele MMC Trend', line=dict(color='blue', width=3))
        fig_mmc = h_mmc.make_fig()
        fig_mmc.update_layout(title_text="Model Drift: Multi-Modal Concordance (0 = Stabiel, Negatief = Drift)", yaxis=dict(range=[-4.0, 2.0]))
        fig_mmc.add_hline(y=0, line_dash="dash", line_color="green", opacity=0.8)
        fig_mmc.write_html(os.path.join(args.output_dir, 'dashboard_1_mmc_smoothed.html'))

    if len(results_auroc) > 1:
        df_auc = pd.DataFrame(results_auroc).set_index('Datum')
        df_auc['AUROC_Smoothed'] = df_auc['AUROC_Score'].ewm(span=4, adjust=False).mean()
        # CSV export is weer terug!
        df_auc.to_csv(os.path.join(args.output_dir, 'auroc_results.csv'))
        
        h_auc = FigureHelper(x=df_auc.index)
        h_auc.add_trace(y=df_auc['AUROC_Smoothed'], name='AUROC Trend (Smoothed)', line=dict(color='darkred', width=3))
        fig_auc = h_auc.make_fig()
        
        fig_auc.update_layout(title_text="Model Performance: Echte AUROC (Ground Truth)", yaxis=dict(range=[0.2, 1.05]))
        fig_auc.add_hline(y=0.5, line_dash="dash", line_color="black", opacity=0.5)
        fig_auc.write_html(os.path.join(args.output_dir, 'dashboard_2_auroc.html'))

    # 4B. MAPJE MET METADATA STATS MAKEN
    meta_dir = os.path.join(args.output_dir, 'metadata_stats')
    os.makedirs(meta_dir, exist_ok=True)
    
    if tracking_ks:
        pd.DataFrame(tracking_ks).set_index('Datum').to_csv(os.path.join(meta_dir, 'metadata_continuous_ks.csv'))
    if tracking_chi2:
        pd.DataFrame(tracking_chi2).set_index('Datum').to_csv(os.path.join(meta_dir, 'metadata_categorical_chi2.csv'))

    print(f" Dashboards klaar!  Metadata drift details zijn opgeslagen in: {meta_dir}")

if __name__ == '__main__': main()

Overwriting MedImaging-ModelDriftMonitoring/src/scripts/drift/generate_drift_csv.py


In [8]:
import sys
import os
import torch

# 1. Zorg dat het notebook de Microsoft broncode map kan vinden
repo_path = os.path.abspath("./MedImaging-ModelDriftMonitoring")
if repo_path not in sys.path:
    sys.path.append(repo_path)

# 2. Importeer de gerepareerde main-functies
from src.scripts.vae.train import main as train_vae
from src.scripts.vae.score import main as score_vae
from src.scripts.finetune.score import main as score_clf
from src.scripts.drift.generate_drift_csv import main as generate_drift

print(f" Schone start. PyTorch {torch.__version__} is actief.")
print("---  Starten van de Modulaire Pipeline ---")


ECHTE_TSV = "pneumo_dataset_ITI_rev.tsv"

print("\n[1/4] VAE Baseline Trainen...")
sys.argv = ["train.py", "--images-dir", "./test_beelden", "--tsv-file", ECHTE_TSV, "--output-dir", "./pipeline_data", "--epochs", "50", "--batch-size", "4"]
train_vae()

print("\n[2/4] Visuele Scores Extraheren (VAE)...")
sys.argv = ["score.py", "--images-dir", "./test_beelden", "--tsv-file", ECHTE_TSV, "--model-path", "./pipeline_data/vae_weights.ckpt", "--output-dir", "./pipeline_data", "--batch-size", "4"]
score_vae()

print("\n[3/4] Klinische Scores Extraheren (DenseNet)...")
sys.argv = ["score.py", "--images-dir", "./test_beelden", "--tsv-file", ECHTE_TSV, "--output-dir", "./pipeline_data", "--batch-size", "4"]
score_clf()

print("\n[4/4] Model Drift Berekenen & Dashboard Genereren...")
sys.argv = ["generate-drift-csv.py", "--tsv-file", ECHTE_TSV, "--vae-scores", "./pipeline_data/vae_scores.csv", "--clf-scores", "./pipeline_data/clf_scores.csv", "--weights-file", "MedImaging-ModelDriftMonitoring/models/weights/metric_weights.csv", "--output-dir", "./pipeline_data"]
generate_drift()

print("\n Klaar! Bekijk de ./pipeline_data map voor je interactieve dashboard!")

 Schone start. PyTorch 2.12.0+cu130 is actief.
---  Starten van de Modulaire Pipeline ---

[1/4] VAE Baseline Trainen...


Epoch 1/50: 100%|██████████| 538/538 [00:32<00:00, 16.41it/s]


Epoch 1 Loss: 396415.6232


Epoch 2/50: 100%|██████████| 538/538 [00:31<00:00, 16.92it/s]


Epoch 2 Loss: 396239.4876


...

Epoch 50 Loss: 396112.2073

[2/4] Visuele Scores Extraheren (VAE)...


VAE Extractie: 100%|██████████| 916/916 [00:28<00:00, 32.27it/s]



[3/4] Klinische Scores Extraheren (DenseNet)...


DenseNet Extractie: 100%|██████████| 916/916 [02:43<00:00,  5.62it/s]



[4/4] Model Drift Berekenen & Dashboard Genereren...
 Analyse opstarten... Baseline N=1099, Productie N=2565
 Dashboards klaar!  Metadata drift details zijn opgeslagen in: ./pipeline_data/metadata_stats

 Klaar! Bekijk de ./pipeline_data map voor je interactieve dashboard!


In [ ]:
import os, sys, shutil
import pandas as pd
import numpy as np
from PIL import Image, ImageEnhance
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp, chi2_contingency 
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

# =====================================================================
# Dashboard
# =====================================================================
CONFIG = {
    "fysieke_visuele_drift": True,          # AAN: Beelden manipuleren
    "helderheid_factor": 4.0,               
    "ruis_factor": 0,                      # AAN: Ruis injectie
    "simuleer_concept_drift": False,       
    "simuleer_conventionele_drift": False,   # AAN: Demografie manipuleren
    "drift_start_percentage": 0.80,
    "smoothing_factor": 3                  
}

BRON_MAP = "./test_beelden"
EXP_DIR = os.path.abspath("./pipeline_data/master_experiment")
IMG_DIR = f"{EXP_DIR}/images"
# =====================================================================

print("[1/6] Veilige mappen maken en metadata inladen...")
os.makedirs(IMG_DIR, exist_ok=True)
df_meta = pd.read_csv("pneumo_dataset_ITI_rev.tsv", sep='\t', low_memory=False)
df_meta['StudyDate_DICOM'] = pd.to_datetime(df_meta['StudyDate_DICOM'], format='%Y%m%d', errors='coerce')
df_meta = df_meta.dropna(subset=['StudyDate_DICOM']).sort_values('StudyDate_DICOM').reset_index(drop=True)

split_idx = int(len(df_meta) * CONFIG["drift_start_percentage"])
drift_start_date = df_meta.iloc[split_idx]['StudyDate_DICOM'].strftime('%Y-%m-%d')

print(f"[2/6] Fysieke drift (Ruis) toepassen vanaf {drift_start_date}...")
for i, row in df_meta.iterrows():
    img_name = str(row['ImageID']).replace('.png', '').replace('.jpg', '') + '.png'
    src, dst = os.path.join(BRON_MAP, img_name), os.path.join(IMG_DIR, img_name)
    if os.path.exists(src):
        if i >= split_idx and CONFIG["fysieke_visuele_drift"]:
            img = Image.open(src).convert('RGB')
            img = ImageEnhance.Brightness(img).enhance(CONFIG["helderheid_factor"])
            if CONFIG["ruis_factor"] > 0:
                np_img = np.array(img).astype(np.int16) 
                noise = np.random.normal(0, CONFIG["ruis_factor"], np_img.shape)
                np_img = np.clip(np_img + noise, 0, 255).astype(np.uint8)
                img = Image.fromarray(np_img)
            img.save(dst)
        else:
            shutil.copy(src, dst)

print("[3/6] Oude scores wissen en PyTorch dwingen opnieuw te scoren...")
for f in ["vae_scores.csv", "clf_scores.csv"]:
    oude_score = os.path.join(EXP_DIR, f)
    if os.path.exists(oude_score): os.remove(oude_score)

from src.scripts.vae.score import main as score_vae
from src.scripts.finetune.score import main as score_clf

sys.argv = ["score.py", "--images-dir", IMG_DIR, "--tsv-file", "pneumo_dataset_ITI_rev.tsv", "--model-path", "./pipeline_data/vae_weights.ckpt", "--output-dir", EXP_DIR, "--batch-size", "4"]
try: score_vae() 
except: pass

sys.argv = ["score.py", "--images-dir", IMG_DIR, "--tsv-file", "pneumo_dataset_ITI_rev.tsv", "--output-dir", EXP_DIR, "--batch-size", "4"]
try: score_clf() 
except: pass

print("[4/6] Data samenvoegen &  Drift toevoegen")
df_vae = pd.read_csv(f"{EXP_DIR}/vae_scores.csv", dtype={'ImageID': str})
df_clf = pd.read_csv(f"{EXP_DIR}/clf_scores.csv", dtype={'ImageID': str})
for d in [df_meta, df_vae, df_clf]:
    d['ImageID'] = d['ImageID'].astype(str).str.replace('.png', '', regex=False)

df_live = pd.merge(df_meta, df_vae, on='ImageID', how='inner')
df_live = pd.merge(df_live, df_clf, on='ImageID', how='inner')
df_live = df_live.sort_values('StudyDate_DICOM').reset_index(drop=True)

live_split_idx = int(len(df_live) * CONFIG["drift_start_percentage"])

if CONFIG["simuleer_conventionele_drift"] and 'PatientBirth' in df_live.columns:
    drift_fase = df_live.iloc[live_split_idx:].copy()
    weights_inject = np.where(drift_fase['PatientBirth'] >= 1990, 5000.0, 0.05)
    sampled_df = drift_fase.sample(n=len(drift_fase), replace=True, weights=weights_inject/weights_inject.sum(), random_state=42)
    for col in df_live.columns:
        df_live.loc[df_live.index[live_split_idx:], col] = sampled_df[col].values

num_cols = ['PatientBirth', 'prob_Pneumonia'] + [f'mu.{i:03d}' for i in range(128)]
for col in num_cols:
    if col in df_live.columns: df_live[col] = pd.to_numeric(df_live[col], errors='coerce')

# =====================================================================
# GEWICHTEN INLADEN (Originele Microsoft filter op 'distance')
# =====================================================================
weights_dict = {}
weights_path = "metric_weights (3).csv"
if os.path.exists(weights_path):
    df_w = pd.read_csv(weights_path)
    col_name = 'Unnamed: 0' if 'Unnamed: 0' in df_w.columns else df_w.columns[0]
    
    if 'Unnamed: 2' in df_w.columns:
        df_w = df_w[df_w['Unnamed: 2'] == 'distance']
        
    weights_dict = dict(zip(df_w[col_name], df_w['abs(corr)'] if 'abs(corr)' in df_w.columns else [1.0]*len(df_w)))
    
    if 'activation.Pneumonia' in weights_dict: weights_dict['prob_Pneumonia'] = weights_dict['activation.Pneumonia']
    if 'age' in weights_dict: weights_dict['PatientBirth'] = weights_dict['age']
    if 'PatientSex' in weights_dict: weights_dict['PatientSex_DICOM'] = weights_dict['PatientSex']

    # Wiskundige diagnose
    d_weight = weights_dict.get('prob_Pneumonia', 0)
    v_weight = sum([v for k,v in weights_dict.items() if 'mu.' in k])
    print(f" --- INFORMATIE: DenseNet weegt {d_weight:.2f}, VAE weegt totaal {v_weight:.2f} ---")

print("[5/6] Microsoft Statistische Normalisatie Uitvoeren...")
baseline_idx = int(len(df_live) * 0.30)
df_ref = df_live.iloc[:baseline_idx].copy()

feat_ks = [c for c in df_live.columns if c.startswith('mu.')] + ['prob_Pneumonia', 'PatientBirth']
feat_chi2 = ['PatientSex_DICOM'] 

baseline_stats = {f: {'dists': []} for f in feat_ks + feat_chi2}
FIXED_SAMPLE_SIZE = min(50, len(df_ref) // 3)

for _ in range(50):
    samp1 = df_ref.sample(n=FIXED_SAMPLE_SIZE, replace=True)
    samp2 = df_ref.sample(n=FIXED_SAMPLE_SIZE, replace=True)
    for f in feat_ks:
        stat, _ = ks_2samp(samp1[f].dropna(), samp2[f].dropna())
        baseline_stats[f]['dists'].append(stat)
    for f in feat_chi2:
        c1, c2 = samp1[f].astype(str).value_counts(), samp2[f].astype(str).value_counts()
        cats = list(set(c1.index) | set(c2.index))
        if len(cats) > 1:
            tab = [[c1.get(c, 0) for c in cats], [c2.get(c, 0) for c in cats]]
            stat, _, _, _ = chi2_contingency(tab)
            baseline_stats[f]['dists'].append(stat)

for f in baseline_stats:
    arr = baseline_stats[f]['dists']
    baseline_stats[f]['mean'] = np.mean(arr) if arr else 0.0
    baseline_stats[f]['std'] = max(np.std(arr), 0.001) if arr else 1.0

# =====================================================================
# DRIFT BEREKENEN (Rolling Window, ONBEGRENSDE Z-Scores)
# =====================================================================
results = []
maanden = [g for n, g in df_live.iloc[baseline_idx:].groupby(pd.Grouper(key='StudyDate_DICOM', freq='M'))]

for i in range(len(maanden)):
    # 3 Maanden Rolling Window (Exact zoals de Microsoft Repo)
    if i == 0: win_df = maanden[i]
    elif i == 1: win_df = pd.concat([maanden[i-1], maanden[i]])
    else: win_df = pd.concat([maanden[i-2], maanden[i-1], maanden[i]])
    
    datum = maanden[i]['StudyDate_DICOM'].max()
    if pd.isna(datum) or len(win_df) < 5: 
        continue
    
    maand_z_scores = {}
    
    for f in feat_ks:
        d_win = win_df[f].dropna()
        stats = []
        for _ in range(10):
            if len(d_win) > 5:
                samp = d_win.sample(n=FIXED_SAMPLE_SIZE, replace=True)
                s, _ = ks_2samp(df_ref[f].dropna(), samp)
                stats.append(s)
        if stats:
            z = (np.mean(stats) - baseline_stats[f]['mean']) / baseline_stats[f]['std']
            maand_z_scores[f] = z  # GEEN GRENZEN, crashen toegestaan!
            
    for f in feat_chi2:
        c1, c2 = df_ref[f].astype(str).dropna(), win_df[f].astype(str).dropna()
        if len(c1) > 5 and len(c2) > 5:
            vc1, vc2 = c1.value_counts(), c2.value_counts()
            cats = list(set(vc1.index) | set(vc2.index))
            if len(cats) > 1:
                tab = [[vc1.get(c, 0) for c in cats], [vc2.get(c, 0) for c in cats]]
                stat, _, _, _ = chi2_contingency(tab)
                z = (stat - baseline_stats[f]['mean']) / baseline_stats[f]['std']
                maand_z_scores[f] = z

    def calc_mmc(features, gebruik_gewichten=True):
        teller, noemer = 0.0, 0.0
        for f in features:
            if f in maand_z_scores:
                w = weights_dict.get(f, 0.1) if gebruik_gewichten else 1.0
                teller += w * maand_z_scores[f]
                noemer += w
        return -(teller/noemer) if noemer > 0 else 0

    if not maand_z_scores:
        continue

    totale_mmc_gewogen = calc_mmc(feat_ks + feat_chi2, gebruik_gewichten=True) 
    totale_mmc_ongewogen = calc_mmc(feat_ks + feat_chi2, gebruik_gewichten=False) 
    
    vae_mmc = calc_mmc([c for c in df_live.columns if c.startswith('mu.')], gebruik_gewichten=True)
    densenet_mmc = calc_mmc(['prob_Pneumonia'], gebruik_gewichten=True)
    meta_mmc = calc_mmc(['PatientBirth', 'PatientSex_DICOM'], gebruik_gewichten=True)
    
    y_true = win_df['Labels'].apply(lambda x: 1 if str(x).strip() == '1' or 'pneumonia' in str(x).lower() else 0)
    auc_score = roc_auc_score(y_true, win_df['prob_Pneumonia'].astype(float)) if len(y_true.unique()) > 1 else 0.85
    
    results.append({
        'Datum': datum, 'MMC_Gewogen': totale_mmc_gewogen, 'MMC_Ongewogen': totale_mmc_ongewogen,
        'VAE_MMC': vae_mmc, 'Dense_MMC': densenet_mmc, 'Meta_MMC': meta_mmc, 'AUROC': auc_score
    })

df_res = pd.DataFrame(results).set_index('Datum')
for col in ['MMC_Gewogen', 'MMC_Ongewogen', 'VAE_MMC', 'Dense_MMC', 'Meta_MMC', 'AUROC']: 
    df_res[f'{col}_Smoothed'] = df_res[col].ewm(span=CONFIG["smoothing_factor"], adjust=False).mean()

print("[6/6] Dashboard genereren...")

fig = make_subplots(
    rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=(
        "<b>1. AUROC, MMCw, MMC0</b>", 
        "<b>2. VAE soft predictions</b>",
        "<b>3. Classifier predictions</b>",
        "<b>4. Meta data</b>"
    ),
    specs=[[{"secondary_y": True}], [{"secondary_y": False}], [{"secondary_y": False}], [{"secondary_y": False}]]
)

fig.add_trace(go.Scatter(x=df_res.index, y=df_res['MMC_Gewogen_Smoothed'], name="MMCw", line=dict(color='purple', width=1), legend="legend"), row=1, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=df_res.index, y=df_res['MMC_Ongewogen_Smoothed'], name="MMC0", line=dict(color='gray', width=1, dash='dot'), legend="legend"), row=1, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=df_res.index, y=df_res['AUROC_Smoothed'], name="AUROC", line=dict(color='#8b0000', width=1), legend="legend"), row=1, col=1, secondary_y=True)

fig.add_trace(go.Scatter(x=df_res.index, y=df_res['VAE_MMC_Smoothed'], name="VAE soft predictions", line=dict(color='#1f77b4', width=1), legend="legend2"), row=2, col=1)
fig.add_trace(go.Scatter(x=df_res.index, y=df_res['Dense_MMC_Smoothed'], name="Classifier predictions", line=dict(color='#ff7f0e', width=1), legend="legend3"), row=3, col=1)
fig.add_trace(go.Scatter(x=df_res.index, y=df_res['Meta_MMC_Smoothed'], name="Meta data", line=dict(color='magenta', width=1), legend="legend4"), row=4, col=1)

for row_idx in range(1, 5):
    fig.add_vline(x=drift_start_date, line_width=3, line_dash="dot", line_color="black", row=row_idx, col=1, annotation_text="Start Extreme Drift" if row_idx == 1 else None)

fig.update_layout(
    title_text=f"<b>Real-World MLOps Diagnose (Smoothing Factor: {CONFIG['smoothing_factor']})</b>", 
    hovermode="x unified", 
    height=2000, 
    showlegend=True,
    legend=dict(y=0.905, x=1.05, yanchor="middle", xanchor="left", bgcolor="rgba(255, 255, 255, 0.8)", bordercolor="Black", borderwidth=1, title_text="<b>AUROC, MMCw, MMC0</b>"),
    legend2=dict(y=0.635, x=1.05, yanchor="middle", xanchor="left", bgcolor="rgba(255, 255, 255, 0.8)", bordercolor="Black", borderwidth=1, title_text="<b>VAE soft predictions</b>"),
    legend3=dict(y=0.365, x=1.05, yanchor="middle", xanchor="left", bgcolor="rgba(255, 255, 255, 0.8)", bordercolor="Black", borderwidth=1, title_text="<b>Classifier predictions</b>"),
    legend4=dict(y=0.095, x=1.05, yanchor="middle", xanchor="left", bgcolor="rgba(255, 255, 255, 0.8)", bordercolor="Black", borderwidth=1, title_text="<b>Meta data</b>")
)

# Onbegrensde assen zodat het perfect crasht
for row in range(1, 5):
    fig.update_yaxes(range=[-8.0, 0.5], secondary_y=False, row=row, col=1)
    
fig.update_yaxes(title_text="<b>Z-Score </b>", row=1, col=1)
fig.update_yaxes(title_text="<b>AUROC</b>", range=[0.2, 1.05], secondary_y=True, row=1, col=1)
fig.update_yaxes(title_text="<b>Z-Score </b>", row=2, col=1)
fig.update_yaxes(title_text="<b>Z-Score </b>", row=3, col=1)
fig.update_yaxes(title_text="<b>Z-Score </b>", row=4, col=1)

html_pad = os.path.join(EXP_DIR, "dashboard_final.html")
fig.write_html(html_pad)

print("Grafiek inladen...")
with open(html_pad, "r", encoding="utf-8") as f:
    display(HTML(f.read()))

[1/6] Veilige mappen maken en metadata inladen...
[2/6] Fysieke drift (Ruis) toepassen vanaf 2016-05-27...
[3/6] Oude scores wissen en PyTorch dwingen opnieuw te scoren...


DenseNet Extractie:  86%|████████▌ | 789/916 [02:24<00:24,  5.26it/s]